In [ ]:
# Import from the installed package instead of re-defining everything
# Run: uv pip install -e . first
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL

In [9]:
import httpx
from httpx import Response
from IPython.display import JSON
import logging
import time
import json

OMLX_API_URL = "http://127.0.0.1:8321/v1" # OpenAI API compatible
OMLX_API_KEY = "changeme"
OMLX_MODEL = "Qwen3.6-35B-A3B-4bit"

In [21]:
# logging setup 
import sys
import logging
from typing import Optional, Dict

from colorama import Fore, Back, Style


class ColoredFormatter(logging.Formatter):
    """Colored log formatter."""

    def __init__(self, *args, colors: Optional[Dict[str, str]]=None, **kwargs) -> None:
        """Initialize the formatter with specified format strings."""

        super().__init__(*args, **kwargs)

        self.colors = colors if colors else {}

    def format(self, record) -> str:
        """Format the specified record as text."""

        record.color = self.colors.get(record.levelname, '')
        record.reset = Style.RESET_ALL

        return super().format(record)


formatter = ColoredFormatter(
    '{asctime} |{color} {levelname:8} {reset}| {name} | {message}',
    style='{', datefmt='%Y-%m-%d %H:%M:%S',
    colors={
        'DEBUG': Fore.CYAN,
        'INFO': Fore.GREEN,
        'WARNING': Fore.YELLOW,
        'ERROR': Fore.RED,
        'CRITICAL': Fore.RED + Back.WHITE + Style.BRIGHT,
    }
)

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(formatter)

logger = logging.getLogger()
logger.handlers[:] = []
logger.addHandler(handler)
logger.setLevel(logging.DEBUG)

logger.info("Logger initialized with colored output")

2026-05-22 16:15:09 | INFO     | root | Logger initialized with colored output


In [ ]:
# Calling the LLM
# official docs : 
# https://developers.openai.com/api/reference/overview
# https://docs.ollama.com/api/openai-compatibility
# https://developers.openai.com/api/docs/guides/function-calling?api-mode=chat


# Transform the function into the tool format expected by the LLM
# decorator to convert a function into a tool definition
def _legacy_tool(func):
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": func.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    # self, cls are not included in the arguments list
                    # arg type can be inferred from type hints if available, otherwise default to string
                    arg: {"type": "string"} for arg in func.__code__.co_varnames[:func.__code__.co_argcount] if arg not in ('self', 'cls')
                },
                # ignore self, cls and any *args, **kwargs
                "required": [arg for arg in func.__code__.co_varnames[:func.__code__.co_argcount] if arg not in ('self', 'cls') and not arg.startswith('*')]
            }
        }
    }

import inspect
from datetime import datetime

def function_to_json(func) -> dict:
    """
    Converts a Python function into a JSON-serializable dictionary
    that describes the function's signature, including its name,
    description, and parameters.

    Args:
        func: The function to be converted.

    Returns:
        A dictionary representing the function's signature in JSON format.
    """
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }

    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {str(e)}"
        )

    # exclude self, cls, and any parameters that are not explicitly defined in the function signature
    parameters = {}
    required = []
    for param in signature.parameters.values():
        if param.name in ('self', 'cls'):
            continue
        try:
            param_type = type_map.get(param.annotation, "string")
        except KeyError as e:
            raise KeyError(
                f"Unknown type annotation {param.annotation} for parameter {param.name}: {str(e)}"
            )
        parameters[param.name] = {"type": param_type}

        if param.default == inspect._empty and not param.name.startswith('*') and not param.name.startswith('**') and param.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD) and param.name not in ('self', 'cls'):
            required.append(param.name)
    
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": func.__doc__ or "",
            "parameters": {
                "type": "object",
                "properties": parameters,
                "required": required,
            },
        },
    }


def tool(func):
    """Convert a Python function into a tool definition for the LLM."""
    return function_to_json(func)

class ToolsRegistry:
    @staticmethod
    def get_weather(location: str) -> str:
        """Get the current weather for a location."""
        return f"The current weather in {location} is sunny with a temperature of 25 degrees Celsius."
    
tool(ToolsRegistry.get_weather)

{'type': 'function',
 'function': {'name': 'get_weather',
  'description': ' Get the current weather for a location. ',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string'}},
   'required': ['location']}}}

In [28]:

class LLM:
    def __init__(self, api_url=OMLX_API_URL, api_key=OMLX_API_KEY, model=OMLX_MODEL):
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.setup_session()
        self.available_models()
    
    def setup_session(self):
        self.session = httpx.Client()
        self.session.headers.update({
            "content-type": "application/json",
            "Authorization": "Bearer " + self.api_key
        })
        self.session.timeout = 30.0
    
    def available_models(self):
        resp = self.session.get(self.api_url+'/models')
        data = resp.json()
        models = [model['id'] for model in data.get('data', [])]
        logger.info("Available models: %s", models)
        return models
    
    def chat_completion(self, messages, tools=[]):
        logger.info("Request sent to %s with model %s", self.api_url+'/chat/completions', self.model)
        resp = self.session.post(self.api_url+'/chat/completions', json={
            "model": self.model,
            "messages": messages,
            "tools": tools
        })
        logger.info("Received response: %s (%d bytes)", resp.status_code, len(resp.content))
        logger.debug("Response content: %s", resp.json())
        return resp

    def execute_tool_calls(self, message):
        tool_calls = message.get('tool_calls', [])
        results = []
        for tool_call in tool_calls or []:
            if tool_call['type'] == 'function':
                func_name = tool_call['function']['name']
                args = tool_call['function']['arguments']
                args = json.loads(args) if isinstance(args, str) else args
                logger.info("Processing tool call: %s with args %s ", func_name, args)
                ToolsClass = ToolsRegistry()
                if hasattr(ToolsClass, func_name):
                    func = getattr(ToolsClass, func_name)
                    result = func(**args) # execute the tool function with the provided arguments
                    results.append({
                        "role": "tool",
                        "tool_call_id": tool_call['id'],
                        "content": result
                    })
                    logger.info("Executed tool: %s with args %s", func_name, args)
                else:
                    logger.warning("Tool function %s not found in Tools class", func_name)
            else:
                logger.warning("Non-function tool calls are not supported in this implementation")
        return results

llm = LLM()

messages = [
    {
        "role": "system", 
        "content": "You are a helpful assistant. Answer the user's question based on the tools you have. Here are the Python tools you have access to: 1. get_weather(location) - Get the current weather for a given location."
    },
    {
        "role": "user", 
        "content": "What's the weather like in New York City today?"
    }
]
print("messages:", json.dumps(messages, indent=4))

# 0. Define the tools that the LLM can use based on the functions in the Tools class
tools = [tool(ToolsRegistry.get_weather)]
print("tools:", json.dumps(tools, indent=4))
time.sleep(1)

# 1. Ask the LLM a question, and tell to answer based on the tools it has access to.
# 2. LLM responds with a message that includes a tool call
resp = llm.chat_completion(messages, tools=tools)
message = resp.json().get('choices', [{}])[0].get('message', {})
print("message:", json.dumps(message, indent=4))
messages.append(message)
time.sleep(1)

# 3. Execute any tool calls that the LLM requested for the message
results = llm.execute_tool_calls(message)
print("results:", json.dumps(results, indent=4))
# 4. Add the tool results back to the messages list as tool responses
messages.extend(results)
print("messages:", json.dumps(messages, indent=4))
time.sleep(1)

# 5. Send the updated messages back to the LLM to get a final response that includes the tool results
final_resp = llm.chat_completion(messages)
final_message = final_resp.json().get('choices', [{}])[0].get('message', {})
print("final_message:", json.dumps(final_message, indent=4))



2026-05-22 16:21:30 | DEBUG    | httpcore.connection | connect_tcp.started host='127.0.0.1' port=8321 local_address=None timeout=30.0 socket_options=None
2026-05-22 16:21:30 | DEBUG    | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x110bf14c0>
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | send_request_headers.started request=<Request [b'GET']>
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | send_request_headers.complete
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | send_request_body.started request=<Request [b'GET']>
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | send_request_body.complete
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | receive_response_headers.started request=<Request [b'GET']>
2026-05-22 16:21:30 | DEBUG    | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'date', b'Fri, 22 May 2026 14:21:29 GMT'), (b'server', b'uvicorn'), (b'content-length', b'

In [ ]:

class Agent(LLM):
    def __init__(self, api_url=OMLX_API_URL, api_key=OMLX_API_KEY, model=OMLX_MODEL, messages=[], tools=[]):
        super().__init__(api_url, api_key, model)
        self.messages = messages
        self.tools = tools
        logger.info("Agent initialized with API URL: %s, Model: %s", api_url, model)

    def step(self):
        resp = self.chat_completion(self.messages, tools=self.tools)
        message = resp.json().get('choices', [{}])[0].get('message', {})
        self.messages.append(message)
        results = self.execute_tool_calls(message)
        self.messages.extend(results)
        final_resp = self.chat_completion(self.messages)
        return final_resp

    def run(self, max_steps=5):
        logger.info("Agent run started with max_steps: %d", max_steps)
        step_num = 0
        while True:
            logger.info("Running Agent step %d/%d", step_num+1, max_steps)
            resp = self.step()
            finish_reason = resp.json().get('choices', [{}])[0].get('finish_reason', '').lower()
            if finish_reason in ('stop', 'exit', 'quit'):
                logger.warning("Agent run stopped by LLM response: %s", finish_reason)
                break
            if step_num >= max_steps:
                logger.warning("Agent run stopped after reaching max_steps: %d", max_steps)
                break
            step_num += 1
        logger.info("Agent run completed with final messages: %s", self.messages)
        return self.messages

messages = [
    {
        "role": "system", 
        "content": "You are a helpful assistant. Answer the user's question based on the tools you have. Here are the Python tools you have access to: 1. get_weather(location) - Get the current weather for a given location."
    },
    {
        "role": "user", 
        "content": "What's the weather like in New York City today?"
    }
]

tools = [tool(ToolsRegistry.get_weather)]

agent = Agent(messages=messages, tools=tools)
messages = agent.run()
print("Final messages:", json.dumps(messages, indent=4))

2026-05-22 13:45:49 | INFO     | root | HTTP session initialized with API key and content type
2026-05-22 13:45:49 | DEBUG    | httpcore.connection | connect_tcp.started host='127.0.0.1' port=8321 local_address=None timeout=30.0 socket_options=None
2026-05-22 13:45:49 | DEBUG    | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x1131b9ca0>
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | send_request_headers.started request=<Request [b'GET']>
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | send_request_headers.complete
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | send_request_body.started request=<Request [b'GET']>
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | send_request_body.complete
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | receive_response_headers.started request=<Request [b'GET']>
2026-05-22 13:45:49 | DEBUG    | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK'